# Bacteria movement: run and tumble
## session 1 of the course 

In [ ]:
# --------------------------
# Python libraries
# --------------------------
import sys
import numpy as np # numerical computing
import matplotlib.pyplot as plt # plotting
import pandas as pd # data handling
from pathlib import Path # file handling

import scipy.io as sio
import os


sys.path.append(os.path.abspath('../sandbox'))
import metrics # custom metrics for run and tumble analysis

import warnings
warnings.filterwarnings("ignore")

# the code below autoloads new changes (no restart of the kernel needed)
%reload_ext autoreload
%autoreload 2

In [ ]:
# --------------------------
# Notebook path
# --------------------------
os.getcwd() 

#### observation:
- In previous session, you learned how to load `.csv` data. here we have a `.mat` format which is MATLAB format
- instead we will use scipy.io to handle this MATLAB files (.mat). its best suitable for this

In [ ]:
os.getcwd()

In [ ]:
# --------------------------
# loading the data
# --------------------------

base_path = os.path.join("./")
ecoli_path = os.path.join(base_path, "../data", "EcoliTrajectories.mat")

print("Base path:", base_path)
print("E. coli path:", ecoli_path)

ecoli_mat = sio.loadmat(ecoli_path)

print("Keys in E. coli file:", list(ecoli_mat.keys()))

# Looking into the data

In [ ]:
# --------------------------
# exploring the data structure
# --------------------------


all_keys = list(ecoli_mat.keys())
dataset_keys = [k for k in all_keys if k.startswith('V_')]

dataset_keys

In [ ]:
# --------------
# parameters to be used here
# -------------

angle_threshold = 45 # angle change threshold for run/tumble classification (degrees)
V_10min = ecoli_mat['V_10min'] # our dataset for now
ref_t = np.logspace(-1.5, 1.5, 100)
# The crossover between the two happens at τ_c (correlation time).
tau_c = 1.6  # τ_c ≈ 1.6 s for wild-type E. coli (from lecture).

In [ ]:
fps = float(V_10min['Parameters'][0, 0]['fps'][0, 0][0, 0])
dt  = 1.0 / fps
n_traj = len(V_10min['Speeds'][0, 0])
n_traj

In [ ]:
lengths = [len(V_10min['Speeds'][0,0][i][0]) for i in range(n_traj)]

plt.hist(lengths, bins=20, edgecolor='black')
plt.title("Distribution of Trajectory Lengths/durations")
plt.xlabel("Number of frames")
plt.ylabel("Count")
plt.show()

In [ ]:
raw = V_10min['Speeds'][0, 0][0][0]

for col in range(raw.shape[1]):
    n_nan = np.sum(np.isnan(raw[:, col]))
    print(f"    col {col}: {n_nan} NaN frames")

In [ ]:
# clean: keep only rows where ALL key columns are finite
key_cols  = [1, 2, 3, 5, 6, 7, 8, 9]   # x, y, z, speed, angle
valid     = np.all(np.isfinite(raw[:, key_cols]), axis=1)
n_dropped = (~valid).sum()
raw_clean = raw[valid]

In [ ]:
raw.shape[0], raw_clean.shape[0]

In [ ]:
df = pd.DataFrame({
    'frame'           : raw_clean[:, 0].astype(int),
    'time_s'          : raw_clean[:, 0] / fps,
    'x_um'            : raw_clean[:, 1],
    'y_um'            : raw_clean[:, 2],
    'z_um'            : raw_clean[:, 3],
    'vx_um_s'         : raw_clean[:, 5],  
    'vy_um_s'         : raw_clean[:, 6],  
    'vz_um_s'         : raw_clean[:, 7],  
    'speed_um_s'      : raw_clean[:, 8],  
    'angle_change_deg': raw_clean[:, 9],  
}).reset_index(drop=True)


In [ ]:
df

# task 1

In [ ]:
'''
compute the step length L for each run segment, 
and collect all L values in a list.
'''

step_lengths = []

for i in range(len(V_10min['Speeds'][0, 0])):
    df, fps = metrics.extract_clean(V_10min, i)
    if len(df) < 10: continue
    df['is_run'] = df['angle_change_deg'] < angle_threshold
    df['seg']    = (df['is_run'] != df['is_run'].shift(1)).cumsum()

    for _, grp in df[df['is_run']].groupby('seg'):
        if len(grp) < 2: continue
        dx = grp['x_um'].iloc[-1] - grp['x_um'].iloc[0]
        dy = grp['y_um'].iloc[-1] - grp['y_um'].iloc[0]
        dz = grp['z_um'].iloc[-1] - grp['z_um'].iloc[0]
        L  = np.sqrt(dx**2 + dy**2 + dz**2)
        if L > 0.1:
            step_lengths.append(L)

step_lengths = np.array(step_lengths)

In [ ]:
'''
plot the histogram of step lengths 
'''

plt.hist(step_lengths, bins=30, edgecolor='black', color='skyblue')
plt.xlabel("step length")
plt.ylabel("count")
plt.show()

The empirical survival function is just the fraction of data points greater than each value. For N sorted data points $L_1 \le L_2 \le ... \le L_N$:

$$P(L>l_i) = 1 - \frac{i}{N} $$

In [ ]:
'''
plot the survival function of step lengths on a semi-log plot,
and compare to an exponential distribution with the same mean.
'''

step_lengths = np.array(step_lengths)
sorted_L     = np.sort(step_lengths)
i = np.arange(1, len(sorted_L)+1)
survival     = 1.0 - i / len(sorted_L)
print(f"i)  N step lengths: {len(step_lengths)},  mean = {step_lengths.mean():.2f} µm")

# ---------- semi-log plot ----------
plt.figure(figsize=(7, 5), facecolor='white')

plt.semilogy(sorted_L, survival, 'o', color='#4A90D9', ms=2.5, alpha=0.5, label='data') # plot data points

lam = 1.0 / step_lengths.mean()
L_fit = np.linspace(sorted_L[0], np.percentile(sorted_L, 97), 300)
plt.semilogy(L_fit, np.exp(-lam * L_fit), '--', color='#D85A30', lw=2.5, label=f'exponential  λ={lam:.2f}') # plot exponential fit
plt.xlabel('step length  L  (µm)', fontsize=11)
plt.ylabel('P(L > l)', fontsize=11)
plt.legend(fontsize=9, framealpha=0.9, edgecolor='#ccc')
plt.show()


the curve is (along) straight on semi-log indicating exponential

In [ ]:
'''
plot the survival function of step lengths on a log-log plot
'''

print("ii) Is the curve straight on semi-log? →", end=" ")
print("yes = exponential")

# ---------- log-log plot ----------
plt.figure(figsize=(7, 5), facecolor='white')

plt.loglog(sorted_L, survival, 'o', color='#4A90D9', ms=2.5, alpha=0.5, label='data') # plot data points
plt.loglog(L_fit, np.exp(-lam * L_fit), '--', color='#D85A30', lw=2.5, label='exponential fit') # plot exponential fit
# straight reference line (power law would look like this)
pl_ref = (sorted_L[0] / L_fit) ** 1.5
plt.loglog(L_fit, pl_ref * survival[0], ':', color='#888', lw=1.5, label='power law reference (straight)')
plt.xlabel('step length  L  (µm)', fontsize=11)
plt.ylabel('P(L > l)', fontsize=11)
plt.legend(fontsize=9, framealpha=0.9, edgecolor='#ccc')

plt.show()

# Task 2


#### Step lengths across time conditions
*  Compute step lengths for V_3min, V_10min, V_50min.
* instead of writing duplicated codes for Computing V_3min, V_10min, V_50min, we use the function below

In [ ]:
'''
function to compute step lengths
'''

def get_steps(V_data):
    """
    Return arrays of (step_length_3d, step_length_2d, mean_speed) per run.
    input: V_data (dataset)
    output: L3: array of 3D step lengths
            L2: array of 2D step lengths
            spd: array of mean speeds

    """
    L3, L2, spd = [], [], []
    for i in range(len(V_data['Speeds'][0, 0])):
        df, fps = metrics.extract_clean(V_data, i)
        if len(df) < 10: continue
        df['is_run'] = df['angle_change_deg'] < angle_threshold
        df['seg']    = (df['is_run'] != df['is_run'].shift(1)).cumsum()
        for _, grp in df[df['is_run']].groupby('seg'):
            if len(grp) < 2: continue
            dx = grp['x_um'].iloc[-1] - grp['x_um'].iloc[0]
            dy = grp['y_um'].iloc[-1] - grp['y_um'].iloc[0]
            dz = grp['z_um'].iloc[-1] - grp['z_um'].iloc[0]
            l3 = np.sqrt(dx**2 + dy**2 + dz**2)
            l2 = np.sqrt(dx**2 + dy**2)
            if l3 > 0.1:
                L3.append(l3)
                L2.append(l2)
                spd.append(grp['speed_um_s'].mean())
    return np.array(L3), np.array(L2), np.array(spd)

definition of L3 and L2 above:
$$L3 = \sqrt{dx^2 + dy^2 + dz^2}$$
$$L2 = \sqrt{dx^2 + dy^2}$$

In [ ]:
'''
function to compute survival function
'''

def survival(L):
    s = np.sort(L)
    p = 1.0 - np.arange(1, len(s)+1) / len(s)
    return s, p

conditions = {'V_3min': '#F0C060', 
              'V_10min': '#1D9E75', 
              'V_50min': '#D85A30'}

plt.figure(figsize=(7, 5), facecolor='white')

for name, color in conditions.items():
    L3, _, _ = get_steps(ecoli_mat[name])
    s, p = survival(L3)
    plt.loglog(s, p, '.', color=color, ms=5.5, alpha=0.5,label=f'{name}')

plt.xlabel('step length  L  (µm)', fontsize=11)
plt.ylabel('P(L > l)', fontsize=11)
plt.title('a) Step lengths across time conditions\nlog-log', fontsize=11, pad=8)
plt.legend(fontsize=9, framealpha=0.9, edgecolor='#ccc')
plt.show()

the curves differ across conditions, indicating that step lengths are not stable over time.

In [ ]:
'''
function to compute μ for an exponential distribution
'''

def mle_mu(L):
    return 1 + len(L) / np.sum(np.log(L / L.min()))

mus = {}
for name, color in conditions.items():
    L3, _, _ = get_steps(ecoli_mat[name])
    mus[name] = mle_mu(L3)
    print(f"  {name}: μ̂ = {mus[name]:.2f}")

spread = max(mus.values()) - min(mus.values())
print(f"\nSpread in μ̂ across conditions: {spread:.2f}")
print("If spread < 0.5 → step lengths stable over time")
print("If spread > 0.5 → cells adapt their movement over time")

# ---- end of code ----